# 📊 MASTER EDA: Toàn Bộ Data Inventory & Train/Test Split Planning

> **Project:** Anti-Face Deepfake Detection (Meta DINOv3 ViT & ConvNeXt)  
> **Workspace Data Root:** `/workspace/data/`  
> **Date:** 2026-08-24  
> **Author / Agent:** Pair Programming Assistant & Hoang Tuan  

---

## 🎯 Mục Tiêu Notebook
Notebook này tổng hợp toàn bộ quá trình **Khám phá Dữ liệu (EDA)**, **Kiểm toán nguồn dữ liệu (Data Inventory)**, và **Kế hoạch phân chia tập huấn luyện/kiểm thử (Train/Test Split Planning)** cho toàn bộ 12+ nguồn dữ liệu deepfake trong workspace.


## 0. Môi Trường & Thư Viện


In [ ]:
import os
import sys
import glob
import json
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Tự động hỗ trợ cả khi có hoặc không có seaborn
try:
    import seaborn as sns
    has_seaborn = True
except ImportError:
    has_seaborn = False

warnings.filterwarnings('ignore')

# Cấu hình visualization
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 200,
    'font.size': 10.5,
    'axes.labelsize': 11,
    'axes.titlesize': 13,
    'xtick.labelsize': 9.5,
    'ytick.labelsize': 9.5,
    'legend.fontsize': 10,
    'figure.titlesize': 14
})

DATA_ROOT = Path('/workspace/data')
HOANGTUAN_SPLITS = DATA_ROOT / 'hoangtuan_data' / 'splits'
REPO_SPLITS = Path('/workspace/hoangtuan/deepfake-ViT/data/splits')
ZERO_LEAK_DIR = DATA_ROOT / 'zero_leakage_benchmark_fixed'

print(f"✅ Data Root: {DATA_ROOT} (Exists: {DATA_ROOT.exists()})")
print(f"✅ Repo Splits: {REPO_SPLITS} (Exists: {REPO_SPLITS.exists()})")
print(f"📦 Seaborn available: {has_seaborn}")


## 1. Tổng Quan Tất Cả 12 Nguồn Dữ Liệu Trong Workspace


In [ ]:
# Khởi tạo bảng tổng quan Data Inventory
inventory_data = [
    {
        "ID": 1,
        "Source Name": "DF40 Train Extracted",
        "Path": "/workspace/data/DF40_train_extracted/",
        "Disk Size (GB)": 74.0,
        "Total Files": 693336,
        "Real Samples": 0,
        "Fake Samples": 693336,
        "Real %": 0.0,
        "Fake %": 100.0,
        "Data Type": "Extracted Frames",
        "Format": ".png",
        "Methods Count": 31,
        "Primary Use": "Main Fake Training Pool"
    },
    {
        "ID": 2,
        "Source Name": "DF40 Test v3",
        "Path": "/workspace/data/test_data_v3/",
        "Disk Size (GB)": 4.4,
        "Total Files": 30692,
        "Real Samples": 1177,
        "Fake Samples": 29515,
        "Real %": 3.84,
        "Fake %": 96.16,
        "Data Type": "Extracted Frames",
        "Format": ".png",
        "Methods Count": 40,
        "Primary Use": "40-Method Test Benchmark"
    },
    {
        "ID": 3,
        "Source Name": "DF40 Test Full",
        "Path": "/workspace/data/df-40-test-full/",
        "Disk Size (GB)": 49.0,
        "Total Files": 100000,
        "Real Samples": 0,
        "Fake Samples": 100000,
        "Real %": 0.0,
        "Fake %": 100.0,
        "Data Type": "Raw Test Archives/Frames",
        "Format": ".png",
        "Methods Count": 38,
        "Primary Use": "Raw Test Pool Benchmark"
    },
    {
        "ID": 4,
        "Source Name": "Celeb-DF v1 (Videos)",
        "Path": "/workspace/data/Celeb-DF/",
        "Disk Size (GB)": 2.1,
        "Total Files": 1203,
        "Real Samples": 408,
        "Fake Samples": 795,
        "Real %": 33.9,
        "Fake %": 66.1,
        "Data Type": "Raw Videos",
        "Format": ".mp4",
        "Methods Count": 1,
        "Primary Use": "Video Benchmark / Frame Extraction"
    },
    {
        "ID": 5,
        "Source Name": "Celeb-DF v2 (Videos)",
        "Path": "/workspace/data/Celeb-DF-v2/",
        "Disk Size (GB)": 9.5,
        "Total Files": 6530,
        "Real Samples": 890,
        "Fake Samples": 5639,
        "Real %": 13.6,
        "Fake %": 86.4,
        "Data Type": "Raw Videos",
        "Format": ".mp4",
        "Methods Count": 1,
        "Primary Use": "Video Benchmark / Frame Extraction"
    },
    {
        "ID": 6,
        "Source Name": "FaceForensics++ (c23)",
        "Path": "/workspace/data/FaceForensics++/",
        "Disk Size (GB)": 2.9,
        "Total Files": 31949,
        "Real Samples": 31949,
        "Fake Samples": 0,
        "Real %": 100.0,
        "Fake %": 0.0,
        "Data Type": "Extracted Frames",
        "Format": ".png",
        "Methods Count": 1,
        "Primary Use": "Primary Real Training/Val Source"
    },
    {
        "ID": 7,
        "Source Name": "DeepFakeFace (DFF)",
        "Path": "/workspace/data/deepFaceFake/",
        "Disk Size (GB)": 5.0,
        "Total Files": 120000,
        "Real Samples": 30000,
        "Fake Samples": 90000,
        "Real %": 25.0,
        "Fake %": 75.0,
        "Data Type": "Face Images (Diffusion)",
        "Format": ".jpg",
        "Methods Count": 3,
        "Primary Use": "Diffusion & Inpainting Training"
    },
    {
        "ID": 8,
        "Source Name": "Deep-Fake-Face-Swap",
        "Path": "/workspace/data/deep-fake-face-swap/",
        "Disk Size (GB)": 0.19,
        "Total Files": 10096,
        "Real Samples": 5048,
        "Fake Samples": 5048,
        "Real %": 50.0,
        "Fake %": 50.0,
        "Data Type": "Face Crops / Parquet",
        "Format": ".jpg / .parquet",
        "Methods Count": 1,
        "Primary Use": "Face Swap Augmentation"
    },
    {
        "ID": 9,
        "Source Name": "Kaggle AI Faces",
        "Path": "/workspace/data/kaggle/",
        "Disk Size (GB)": 7.7,
        "Total Files": 241914,
        "Real Samples": 70000,
        "Fake Samples": 71530,
        "Real %": 49.5,
        "Fake %": 50.5,
        "Data Type": "Face Images (FFHQ + AI)",
        "Format": ".jpg",
        "Methods Count": 4,
        "Primary Use": "Real FFHQ & MidJourney/SFHQ Boost"
    },
    {
        "ID": 10,
        "Source Name": "CelebV-HQ Videos",
        "Path": "/workspace/data/celebvhq/",
        "Disk Size (GB)": 40.0,
        "Total Files": 35666,
        "Real Samples": 35666,
        "Fake Samples": 0,
        "Real %": 100.0,
        "Fake %": 0.0,
        "Data Type": "Raw High-Res Videos",
        "Format": ".mp4",
        "Methods Count": 1,
        "Primary Use": "High-Quality Real Video Pool"
    },
    {
        "ID": 11,
        "Source Name": "CelebV-HQ Frames",
        "Path": "/workspace/data/celebvhq_frames/",
        "Disk Size (GB)": 0.95,
        "Total Files": 24000,
        "Real Samples": 24000,
        "Fake Samples": 0,
        "Real %": 100.0,
        "Fake %": 0.0,
        "Data Type": "Extracted Frames",
        "Format": ".jpg",
        "Methods Count": 1,
        "Primary Use": "Real Frame Supplement"
    },
    {
        "ID": 12,
        "Source Name": "Celeb-DF Processed Frames",
        "Path": "/workspace/data/hoangtuan_data/processed/",
        "Disk Size (GB)": 1.9,
        "Total Files": 39042,
        "Real Samples": 25854,
        "Fake Samples": 13188,
        "Real %": 66.2,
        "Fake %": 33.8,
        "Data Type": "Extracted Face Crops",
        "Format": ".png",
        "Methods Count": 1,
        "Primary Use": "Secondary Real/Fake Training Pool"
    }
]

df_inventory = pd.DataFrame(inventory_data)
display(df_inventory[["ID", "Source Name", "Disk Size (GB)", "Total Files", "Real Samples", "Fake Samples", "Real %", "Fake %", "Data Type", "Primary Use"]])


### Biểu Đồ 1: Phân Bổ Dung Lượng (GB) và Số Lượng File Giữa Các Nguồn


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_sorted_disk = df_inventory.sort_values(by="Disk Size (GB)", ascending=True)
if has_seaborn:
    colors_disk = sns.color_palette("mako", len(df_sorted_disk))
else:
    colors_disk = plt.cm.viridis(np.linspace(0.2, 0.9, len(df_sorted_disk)))

bars1 = axes[0].barh(df_sorted_disk["Source Name"], df_sorted_disk["Disk Size (GB)"], color=colors_disk, edgecolor='black', linewidth=0.7)
axes[0].set_title("Dung Lượng Ổ Đĩa Từng Nguồn Data (GB)", fontweight='bold', fontsize=12)
axes[0].set_xlabel("Dung lượng (GB)")
for bar in bars1:
    width = bar.get_width()
    axes[0].text(width + 1.0, bar.get_y() + bar.get_height()/2, f"{width:.1f} GB", va='center', fontsize=9, fontweight='semibold')

df_sorted_files = df_inventory.sort_values(by="Total Files", ascending=True)
if has_seaborn:
    colors_files = sns.color_palette("viridis", len(df_sorted_files))
else:
    colors_files = plt.cm.plasma(np.linspace(0.2, 0.9, len(df_sorted_files)))

bars2 = axes[1].barh(df_sorted_files["Source Name"], df_sorted_files["Total Files"] / 1e3, color=colors_files, edgecolor='black', linewidth=0.7)
axes[1].set_title("Tổng Số Lượng Files Từng Nguồn Data (Nghìn Files)", fontweight='bold', fontsize=12)
axes[1].set_xlabel("Số lượng files (k = nghìn)")
for bar in bars2:
    width = bar.get_width()
    axes[1].text(width + 5.0, bar.get_y() + bar.get_height()/2, f"{width:.1f}k", va='center', fontsize=9, fontweight='semibold')

plt.tight_layout()
plt.show()
